# Evaluation on LLaMA 3 + Instruction-Tuned using BBEH benchmark for reasoning tasks

In [ ]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.8 MB/s eta 0:00:00


In [ ]:
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model_llama = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [ ]:
tokenizer_llama = AutoTokenizer.from_pretrained(model_llama)
model_llama = AutoModelForCausalLM.from_pretrained(model_llama,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
tokenizer_llama.pad_token = tokenizer_llama.eos_token

In [ ]:
def load_json(path):
  with open(path,"r") as f:
    data = json.load(f)

    df = pd.DataFrame(data['examples'])
    return df

In [ ]:
def get_prompt(instruction,questions):
  prompts=[]
  for i in range(len(questions)):
    prompt = f"{instruction}\n\n{questions[i]}\n\nAnswer:"
    prompts.append(prompt)
  return prompts

In [ ]:
def training(prompts,num_tokens=1):
  predictions=[]
  for i in range(len(prompts)):
    input_tokens = tokenizer_llama(prompts[i],return_tensors="pt")
    input_ids = input_tokens["input_ids"].to("cuda:0")
    attention_mask = input_tokens["attention_mask"].to("cuda:0")
    output_tokens = model_llama.generate(input_ids=input_ids,attention_mask=attention_mask,max_new_tokens=num_tokens,pad_token_id=tokenizer_llama.pad_token_id)
    response = tokenizer_llama.decode(output_tokens[0], skip_special_tokens=True)
    pred_answer = response.split("Answer:")[1].split("Explanation:")[0]
    print(f"Iteration {i}\n")
    print(f"Response: {pred_answer}")
    pred_answer = pred_answer.replace("\n","").strip()
    predictions.append(pred_answer)

  return predictions

## Task Casual Understanding

In [ ]:
dataset = load_json("/content/task_casual_understanding.json")

In [ ]:
dataset

,input,target
0,Question: Lauren and Jane work for the same co...,Yes
1,Question: Lauren and Jane work for the same co...,Yes
2,Question: The receptionist in the philosophy d...,Yes
3,Question: The receptionist in the philosophy d...,No
4,Question: Claire's parents bought her an old c...,Yes
...,...,...
195,Question: A test tube contains a mixture. Sam ...,Yes
196,Question: Reagent X and Y are both added to a ...,No
197,Question: There is a test tube on the left and...,Yes
198,Question: Sam is heating Reagent X in a test t...,Yes


In [ ]:
dataset['input'][0]

"Question: Lauren and Jane work for the same company. They each need to use a computer for work sometimes. Unfortunately, the computer isn't very powerful. If two people are logged on at the same time, it usually crashes. So the company decided to institute an official policy. It declared that Lauren would be the only one permitted to use the computer in the mornings and that Jane would be the only one permitted to use the computer in the afternoons. As expected, Lauren logged on the computer the next day at 9:00 am. But Jane decided to disobey the official policy. She also logged on at 9:00 am. The computer crashed immediately. Did Jane cause the computer to crash?  Reply Yes or No based on the answer the majority of people would give. If you think people would be split roughly 50-50 between Yes and No then reply Ambiguous."

In [ ]:
instruction = """
Instruction: You are an AI that answers causal questions based on what the majority of people would think.
"""

In [ ]:
questions = dataset['input'].values.tolist()

In [ ]:
true_answers = dataset['target'].values.tolist()

In [ ]:
questions[2]

"Question: The receptionist in the philosophy department keeps her desk stocked with pens. The administrative assistants are allowed to take the pens, but faculty members are supposed to buy their own. The administrative assistants typically do take the pens. Unfortunately, so do the faculty members. The receptionist has repeatedly emailed them reminders that only administrative assistants are allowed to take the pens. On Monday morning, one of the administrative assistants encounters Professor Smith walking past the receptionist's desk. Both take pens. Later that day, the receptionist needs to take an important message... but she has a problem. There are no pens left on her desk. Did Professor Smith cause the problem?  Reply Yes or No based on the answer the majority of people would give. If you think people would be split roughly 50-50 between Yes and No then reply Ambiguous."

In [ ]:
prompts = get_prompt(instruction,questions)

In [ ]:
prompts[1]

"\nInstruction: You are an AI that answers causal questions based on what the majority of people would think.\n\n\nQuestion: Lauren and Jane work for the same company. They each need to use a computer for work sometimes. Unfortunately, the computer isn't very powerful. If two people are logged on at the same time, it usually crashes. So the company decided to institute an official policy. It declared that Jane would be the only one permitted to use the computer in the mornings and that Lauren would be the only one permitted to use the computer in the afternoons. As expected, Jane logged on the computer the next day at 9:00 am. But Lauren decided to disobey the official policy. She also logged on at 9:00 am. The computer crashed immediately. Did Jane cause the computer to crash?  Reply Yes or No based on the answer the majority of people would give. If you think people would be split roughly 50-50 between Yes and No then reply Ambiguous.\n\nAnswer:"

In [ ]:
predictions = training(prompts)

Iteration 0

Response:  Yes
Iteration 1

Response:  Yes
Iteration 2

Response:  Yes
Iteration 3

Response:  Yes
Iteration 4

Response:  Yes
Iteration 5

Response:  No
Iteration 6

Response:  No
Iteration 7

Response:  No
Iteration 8

Response:  No
Iteration 9

Response:  Yes
Iteration 10

Response:  No
Iteration 11

Response:  No
Iteration 12

Response:  No
Iteration 13

Response:  No
Iteration 14

Response:  Yes
Iteration 15

Response:  Yes
Iteration 16

Response:  Yes
Iteration 17

Response:  Yes
Iteration 18

Response:  No
Iteration 19

Response:  No
Iteration 20

Response:  No
Iteration 21

Response:  Yes
Iteration 22

Response:  No
Iteration 23

Response:  Yes
Iteration 24

Response:  No
Iteration 25

Response:  Yes
Iteration 26

Response:  No
Iteration 27

Response:  No
Iteration 28

Response:  Yes
Iteration 29

Response:  No
Iteration 30

Response:  Yes
Iteration 31

Response:  No
Iteration 32

Response:  No
Iteration 33

Response:  No
Iteration 34

Response:  No
Iteration 35

R

In [ ]:
true_answers[:10]

['Yes', 'Yes', 'Yes', 'No', 'Yes', 'Ambiguous', 'No', 'Ambiguous', 'Yes', 'No']

In [ ]:
predictions[:10]

['Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'Yes']

In [ ]:
predictions = [p if p=="Yes" or p=="No" else "Ambiguous" for p in predictions]

In [ ]:
predictions[:19]

['Yes',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'No',
 'No',
 'No',
 'No',
 'Yes',
 'Yes',
 'Yes',
 'Yes',
 'No']

In [ ]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

   Ambiguous       0.00      0.00      0.00        45
          No       0.38      0.47      0.42        80
         Yes       0.37      0.39      0.38        75

    accuracy                           0.34       200
   macro avg       0.25      0.29      0.27       200
weighted avg       0.29      0.34      0.31       200



## Task Movie Recommendation

In [ ]:
dataset = load_json("/content/task_movie_recommendation.json")

In [ ]:
dataset

,input,target
0,Which option has more similar movies in terms ...,(E)
1,Which option has more similar movies in terms ...,(C)
2,Which option has more similar movies in terms ...,(C)
3,Which option has more similar movies in terms ...,(A)
4,Which option has more similar movies in terms ...,(I)
...,...,...
195,Which option has more similar movies in terms ...,(E)
196,Which option has more similar movies in terms ...,(E)
197,Which option has more similar movies in terms ...,(E)
198,Which option has more similar movies in terms ...,(A)


In [ ]:
questions = dataset['input'].values.tolist()

In [ ]:
questions[0]

'Which option has more similar movies in terms of whether a group of people will like them or not?\nOptions:\n(A) The Mask, The Lion King, Pretty Woman, Batman, The Front Page\n(B) Maelstrom, The Lion King, Lamerica, Pretty Woman, The Front Page\n(C) The Fugitive, The Mask, Maelstrom, Pretty Woman, Batman\n(D) The Lion King, The Mask, Lamerica, Maelstrom, Batman\n(E) Batman, The Lion King, Pretty Woman, The Fugitive, The Mask\n(F) Batman, The Mask, Maelstrom, The Fugitive, Lamerica\n(G) The Front Page, The Fugitive, Maelstrom, Lamerica, Batman\n(H) The Fugitive, Pretty Woman, Lamerica, Batman, The Lion King\n(I) The Fugitive, Lamerica, The Mask, The Front Page, Batman\n(J) The Fugitive, Pretty Woman, The Lion King, The Mask, The Front Page\n'

In [ ]:
questions[1]

'Which option has more similar movies in terms of whether a group of people will like them or not?\nOptions:\n(A) Street Fighter II The Animated Movie, The Sixth Sense, The Matrix, The Shawshank Redemption, Terminator 2 Judgment Day\n(B) The Boy Who Could Fly, The Sheltering Sky, Forrest Gump, Terminator 2 Judgment Day, The Matrix\n(C) Forrest Gump, The Shawshank Redemption, Terminator 2 Judgment Day, The Matrix, The Sixth Sense\n(D) The Boy Who Could Fly, The Sheltering Sky, Street Fighter II The Animated Movie, The Sixth Sense, Terminator 2 Judgment Day\n(E) The Matrix, The Boy Who Could Fly, Forrest Gump, The Shawshank Redemption, The Sixth Sense\n(F) The Boy Who Could Fly, The Matrix, Terminator 2 Judgment Day, The Sixth Sense, Forrest Gump\n(G) The Shawshank Redemption, Street Fighter II The Animated Movie, The Sheltering Sky, The Matrix, The Boy Who Could Fly\n(H) Street Fighter II The Animated Movie, The Matrix, The Sixth Sense, The Shawshank Redemption, The Sheltering Sky\n(I) 

In [ ]:
true_answers = dataset['target'].values.tolist()

In [ ]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'C', 'A', 'I', 'D', 'B', 'F', 'D', 'H']

In [ ]:
instruction = f"""You are an assistant that determines which group of movies is most similar in terms of what people are likely to enjoy together. Reply only with the letter (A, B, C, D, E, F, G, H, I, or J) of the group that is most similar.
Do not write any explanation, full movie names, or extra text. The answer must be a single uppercase letter."""

In [ ]:
prompts = get_prompt(instruction,questions)

In [ ]:
prompts[1]

'You are an assistant that determines which group of movies is most similar in terms of what people are likely to enjoy together. Reply only with the letter (A, B, C, D, E, F, G, H, I, or J) of the group that is most similar.\nDo not write any explanation, full movie names, or extra text. The answer must be a single uppercase letter.\n\nWhich option has more similar movies in terms of whether a group of people will like them or not?\nOptions:\n(A) Street Fighter II The Animated Movie, The Sixth Sense, The Matrix, The Shawshank Redemption, Terminator 2 Judgment Day\n(B) The Boy Who Could Fly, The Sheltering Sky, Forrest Gump, Terminator 2 Judgment Day, The Matrix\n(C) Forrest Gump, The Shawshank Redemption, Terminator 2 Judgment Day, The Matrix, The Sixth Sense\n(D) The Boy Who Could Fly, The Sheltering Sky, Street Fighter II The Animated Movie, The Sixth Sense, Terminator 2 Judgment Day\n(E) The Matrix, The Boy Who Could Fly, Forrest Gump, The Shawshank Redemption, The Sixth Sense\n(F)

In [ ]:
predictions = training(prompts)

Iteration 0

Response:  B
Iteration 1

Response:  C
Iteration 2

Response:  A
Iteration 3

Response:  F
Iteration 4

Response:  B
Iteration 5

Response:  C
Iteration 6

Response:  A
Iteration 7

Response:  B
Iteration 8

Response:  C
Iteration 9

Response:  F
Iteration 10

Response:  A
Iteration 11

Response:  I
Iteration 12

Response:  C
Iteration 13

Response:  C
Iteration 14

Response:  B
Iteration 15

Response:  B
Iteration 16

Response:  A
Iteration 17

Response:  B
Iteration 18

Response:  C
Iteration 19

Response:  A
Iteration 20

Response:  C
Iteration 21

Response:  C
Iteration 22

Response:  D
Iteration 23

Response:  F
Iteration 24

Response:  B
Iteration 25

Response:  C
Iteration 26

Response:  A
Iteration 27

Response:  B
Iteration 28

Response:  C
Iteration 29

Response:  B
Iteration 30

Response:  A
Iteration 31

Response:  C
Iteration 32

Response:  B
Iteration 33

Response:  C
Iteration 34

Response:  C
Iteration 35

Response:  G
Iteration 36

Response:  B
Iteration 3

In [ ]:
predictions[:10]

['B', 'C', 'A', 'F', 'B', 'C', 'A', 'B', 'C', 'F']

In [ ]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.08      0.15      0.10        26
           B       0.11      0.35      0.17        17
           C       0.08      0.22      0.11        18
           D       0.67      0.10      0.17        20
           E       0.00      0.00      0.00        21
           F       0.22      0.28      0.24        18
           G       0.11      0.05      0.06        22
           H       0.00      0.00      0.00        21
           I       0.20      0.06      0.09        17
           J       0.00      0.00      0.00        20

    accuracy                           0.12       200
   macro avg       0.15      0.12      0.10       200
weighted avg       0.14      0.12      0.09       200



## Task Disambiguation QA

In [ ]:
dataset = load_json("/content/task_disambiguationQA.json")

In [ ]:
dataset

,input,target
0,Here is a sentence with pronoun(s) whose antec...,(A)
1,Here is a sentence with pronoun(s) whose antec...,(D)
2,Here is a sentence with pronoun(s) whose antec...,(E)
3,Here is a sentence with pronoun(s) whose antec...,(B)
4,Here is a sentence with pronoun(s) whose antec...,(A)
...,...,...
115,Here is a sentence with pronoun(s) whose antec...,(D)
116,Here is a sentence with pronoun(s) whose antec...,(A)
117,Here is a sentence with pronoun(s) whose antec...,(B)
118,Here is a sentence with pronoun(s) whose antec...,(D)


In [ ]:
dataset['target'].value_counts()

,count
target,
(B),25
(D),24
(C),23
(A),22
(E),17
(F),7
(G),1
(H),1


In [ ]:
questions = dataset['input'].values.tolist()

In [ ]:
true_answers = dataset['target'].values.tolist()

In [ ]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['A', 'D', 'E', 'B', 'A', 'B', 'B', 'A', 'C', 'D']

In [ ]:
questions[1]

'Here is a sentence with pronoun(s) whose antecedent(s) can either be derived from the context or is ambiguous.\nSentence: The designer called the janitor and asked him to clean the room as soon as possible, but he went to fetch a cup of coffee first and then returned to his desk.\nWhich of the following options correctly explains the antecedent(s) of the pronoun(s)?\n(A) Designer is cleaning the room as soon as possible and got coffee.\n(B) Janitor is cleaning the room as soon as possible and got coffee.\n(C) Designer is cleaning the room as soon as possible and janitor got coffee.\n(D) Janitor is cleaning the room as soon as possible and designer got coffee.\n(E) Ambiguous.'

In [ ]:
instruction = f"""You are an AI assistant solving a pronoun disambiguation task.

For each sentence:
- Identify what the pronoun(s) most likely refer to based on common sense.
- If more than one interpretation is equally plausible, select “Ambiguous”.

Rules:
- Do not assume information that is not stated.
- Do not infer intent beyond what is commonly assumed.
- Choose exactly one answer from (A)–(H) based on available letters.

Final answer format:
(A), (B), (C), (D), (E), (H), (G), or (F)
"""

In [ ]:
prompts = get_prompt(instruction,questions)

In [ ]:
prompts[1]

'You are an AI assistant solving a pronoun disambiguation task.\n\nFor each sentence:\n- Identify what the pronoun(s) most likely refer to based on common sense.\n- If more than one interpretation is equally plausible, select “Ambiguous”.\n\nRules:\n- Do not assume information that is not stated.\n- Do not infer intent beyond what is commonly assumed.\n- Choose exactly one answer from (A)–(H) based on available letters.\n\nFinal answer format:\n(A), (B), (C), (D), (E), (H), (G), or (F)\n\n\nHere is a sentence with pronoun(s) whose antecedent(s) can either be derived from the context or is ambiguous.\nSentence: The designer called the janitor and asked him to clean the room as soon as possible, but he went to fetch a cup of coffee first and then returned to his desk.\nWhich of the following options correctly explains the antecedent(s) of the pronoun(s)?\n(A) Designer is cleaning the room as soon as possible and got coffee.\n(B) Janitor is cleaning the room as soon as possible and got co

In [ ]:
predictions = training(prompts,num_tokens=3)

Iteration 0

Response:  (E)
Iteration 1

Response:  (E)
Iteration 2

Response:  (E)
Iteration 3

Response:  (E)
Iteration 4

Response:  (E)
Iteration 5

Response:  (E)
Iteration 6

Response:  (C)
Iteration 7

Response:  (D)
Iteration 8

Response:  (D)
Iteration 9

Response:  (D)
Iteration 10

Response:  (D)
Iteration 11

Response:  (E)
Iteration 12

Response:  (C)
Iteration 13

Response:  (E)
Iteration 14

Response:  (C)
Iteration 15

Response:  (E)
Iteration 16

Response:  (E)
Iteration 17

Response:  (E)
Iteration 18

Response:  (E)
Iteration 19

Response:  (C)
Iteration 20

Response:  (B)
Iteration 21

Response:  (E)
Iteration 22

Response:  (D)
Iteration 23

Response:  (C)
Iteration 24

Response:  (C)
Iteration 25

Response:  (A)
Iteration 26

Response:  (E)
Iteration 27

Response:  (A)
Iteration 28

Response:  (F)
Iteration 29

Response:  (F)
Iteration 30

Response:  (B)
Iteration 31

Response:  (A),
Iteration 32

Response:  (E)
Iteration 33

Response:  (C)
Iteration 34

Response:

In [ ]:
predictions[:10]

['(E)', '(E)', '(E)', '(E)', '(E)', '(E)', '(C)', '(D)', '(D)', '(D)']

In [ ]:
predictions = [x.strip("()").upper() for x in predictions]
predictions[:10]

['E', 'E', 'E', 'E', 'E', 'E', 'C', 'D', 'D', 'D']

In [ ]:
predictions = [x if x in ['A','B','C','D','E'] else "Ambiguous" for x in predictions]

In [ ]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.45      0.23      0.30        22
   Ambiguous       0.00      0.00      0.00         0
           B       0.44      0.16      0.24        25
           C       0.50      0.65      0.57        23
           D       0.47      0.33      0.39        24
           E       0.23      0.53      0.32        17
           F       0.00      0.00      0.00         7
           G       0.00      0.00      0.00         1
           H       0.00      0.00      0.00         1

    accuracy                           0.34       120
   macro avg       0.23      0.21      0.20       120
weighted avg       0.40      0.34      0.34       120



## Task: Boolean Expressions

In [ ]:
dataset = load_json("/content/task_boolean_expressions.json")

In [ ]:
dataset

,input,target
0,"From the following five expressions, only one ...",(E)
1,"From the following five expressions, only one ...",(C)
2,"From the following five expressions, only one ...",(E)
3,"From the following five expressions, only one ...",(D)
4,"From the following five expressions, only one ...",(C)
...,...,...
195,"From the following five expressions, only one ...",(A)
196,"From the following five expressions, only one ...",(B)
197,"From the following five expressions, only one ...",(B)
198,"From the following five expressions, only one ...",(C)


In [ ]:
dataset["target"].value_counts()

,count
target,
(E),43
(B),41
(C),40
(A),39
(D),37


In [ ]:
questions = dataset["input"].values.tolist()

In [ ]:
true_answers = dataset["target"].values.tolist()

In [ ]:
true_answers = [x.strip("()").upper() for x in true_answers]
true_answers[:10]

['E', 'C', 'E', 'D', 'C', 'D', 'C', 'D', 'A', 'A']

In [ ]:
questions[1]

'From the following five expressions, only one evaluates to true. Which one is it?\n\n(A) not (not not not ((not ((not not (not not not (not not not ((not not not (not not not ((not not not (not (not not not (9 * 9 + -3 * 6 is greater than -9 * -6) and not not (The capital of Azerbaijan is Ganja.)) or (The capital of the Nigeria is Lagos.)) and not not (-10 * -2 <= -1)) and (-1 - (8 / -10) > 1)) and not not (False)) and not not (10 - (-1 / -3) <= -3)) or (The capital of Canada is Toronto.)) and not not (-4 * -3 + -2 * 4 <= -4 * -5)) and not not (-3 * -5 + 2 * 3 is less than or equal to 4 * -9)) or not not not (False)) and (The capital of UAE is Abu Dhabi.)) or (6 * 8 + 4 * 4 is greater than -3 * -3)) and (The capital of the Nigeria is Lagos.)) and not not (4 * 2 <= 7)) and (-8 * 5 + -2 * -7 > -2 * -5)\n(B) not not not ((not (True) or (-6 * -1 + 8 * -2 > 7 * 5)) or (not (The capital of Nepal is Pokhara.) or (True))) and not not (not (not not not (2 * -10 + -2 * -4 > -1 * 1) and not not 

In [ ]:
instruction=f"""You are evaluating boolean expressions.

Exactly one option is True.

Output format rules:
- Output ONLY one of the following exact strings:
(A)
(B)
(C)
(D)
(E)

Do NOT use LaTeX.
Do NOT use special symbols.
Do NOT add explanations.
Do NOT add whitespace or extra characters.
"""

In [ ]:
prompts = get_prompt(instruction,questions)

In [ ]:
prompts[1]

'You are evaluating boolean expressions.\n\nExactly one option is True.\n\nOutput format rules:\n- Output ONLY one of the following exact strings:\n(A)\n(B)\n(C)\n(D)\n(E)\n\nDo NOT use LaTeX.\nDo NOT use special symbols.\nDo NOT add explanations.\nDo NOT add whitespace or extra characters.\n\n\nFrom the following five expressions, only one evaluates to true. Which one is it?\n\n(A) not (not not not ((not ((not not (not not not (not not not ((not not not (not not not ((not not not (not (not not not (9 * 9 + -3 * 6 is greater than -9 * -6) and not not (The capital of Azerbaijan is Ganja.)) or (The capital of the Nigeria is Lagos.)) and not not (-10 * -2 <= -1)) and (-1 - (8 / -10) > 1)) and not not (False)) and not not (10 - (-1 / -3) <= -3)) or (The capital of Canada is Toronto.)) and not not (-4 * -3 + -2 * 4 <= -4 * -5)) and not not (-3 * -5 + 2 * 3 is less than or equal to 4 * -9)) or not not not (False)) and (The capital of UAE is Abu Dhabi.)) or (6 * 8 + 4 * 4 is greater than -3 *

In [ ]:
predictions = training(prompts,num_tokens=3)

Iteration 0

Response:  (B)
Iteration 1

Response:  (A)
Iteration 2

Response:  (C)
Iteration 3

Response:  (E)
Iteration 4

Response:  (C)
Iteration 5

Response:  (E)
Iteration 6

Response:  (A)
Iteration 7

Response:  (A)
Iteration 8

Response:  (A)
Iteration 9

Response:  (E)
Iteration 10

Response:  (D)
Iteration 11

Response:  (B)
Iteration 12

Response:  (A)
Iteration 13

Response:  (D)
Iteration 14

Response:  (A)
Iteration 15

Response:  (A)
Iteration 16

Response:  (E)
Iteration 17

Response:  (C)
Iteration 18

Response:  (A)
Iteration 19

Response:  (E)
Iteration 20

Response:  (B)
Iteration 21

Response:  (A)
Iteration 22

Response:  (C)
Iteration 23

Response:  (C)
Iteration 24

Response:  (D)
Iteration 25

Response:  (E)
Iteration 26

Response:  (C)
Iteration 27

Response:  (B)
Iteration 28

Response:  (E)
Iteration 29

Response:  (A)
Iteration 30

Response:  (C)
Iteration 31

Response:  (D)
Iteration 32

Response:  (E)
Iteration 33

Response:  (D)
Iteration 34

Response: 

In [ ]:
predictions[:10]

['(B)', '(A)', '(C)', '(E)', '(C)', '(E)', '(A)', '(A)', '(A)', '(E)']

In [ ]:
predictions = [x.strip("()").upper() for x in predictions]
predictions[:10]

['B', 'A', 'C', 'E', 'C', 'E', 'A', 'A', 'A', 'E']

In [ ]:
print(classification_report(true_answers,predictions,zero_division=0))

              precision    recall  f1-score   support

           A       0.19      0.21      0.20        39
           B       0.10      0.07      0.09        41
           C       0.15      0.17      0.16        40
           D       0.09      0.05      0.07        37
           E       0.18      0.26      0.21        43

    accuracy                           0.15       200
   macro avg       0.14      0.15      0.15       200
weighted avg       0.14      0.15      0.15       200



##